In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import joblib

# ============================================
# 1. DATA LOADING & PREPROCESSING
# ============================================
df = pd.read_csv('creditcard.csv')
print('Dataset Loaded. Shape:', df.shape)

scaler = StandardScaler()
df['Amount'] = scaler.fit_transform(df[['Amount']])
df = df.drop(['Time'], axis=1)

X = df.drop('Class', axis=1)
y = df['Class']

# ============================================
# 2. TRAIN/TEST SPLIT (Before SMOTE to prevent data leakage)
# ============================================
X_train_orig, X_test, y_train_orig, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

# ============================================
# 3. SMOTE RESAMPLING (Training set only)
# ============================================
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train_orig, y_train_orig)
print('SMOTE Resampling complete.')

# ============================================
# 4. MODEL TRAINING: Random Forest
# ============================================
print('\n--- Training Random Forest ---')
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
print(classification_report(y_test, y_pred_rf))

# ============================================
# 5. MODEL TRAINING: XGBoost
# ============================================
print('\n--- Training XGBoost ---')
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
print(classification_report(y_test, y_pred_xgb))

# ============================================
# 6. MODEL TRAINING: Logistic Regression
# ============================================
print('\n--- Training Logistic Regression ---')
lr_model = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)
print(classification_report(y_test, y_pred_lr))

# ============================================
# 7. SAVE ALL MODEL ARTIFACTS
# ============================================
joblib.dump(rf_model, 'model_random_forest.pkl')
joblib.dump(xgb_model, 'model_xgboost.pkl')
joblib.dump(lr_model, 'model_logistic_regression.pkl')
joblib.dump(scaler, 'scaler.pkl')
print('\nAll 3 models and scaler saved successfully.')
print('  - model_random_forest.pkl')
print('  - model_xgboost.pkl')
print('  - model_logistic_regression.pkl')
print('  - scaler.pkl')